# Making final dataset before tableau stuff begins

# Setup

In [1]:
# imports
import pandas as pd
from idx.components import ingest, preprocessing, feature_engineering, cleaning
from sklearn.pipeline import Pipeline
import geopandas as gpd

In [2]:
# making school district geodataframe
district_gdf = gpd.read_file("../data/raw/district/DistrictAreas2425.shp")
district_gdf = district_gdf.to_crs(epsg=4326)

# initializing pipeline components
ingestor = ingest.MLSIngestor(input_path="../data/raw/")
fred_ingestor = ingest.FredMerger()
cleaner = preprocessing.DataCleaner()
flagger = preprocessing.BadDataFlagger()
market_maker = feature_engineering.CreateMarketMetrics()
cleanup = feature_engineering.CleanUpTransformer()
null = feature_engineering.NullDropper()
dist_merger = feature_engineering.DistrictMerger(district_gdf=district_gdf)
iqr_flagger = cleaning.OutlierFlagger(subset=["OriginalListPrice", "LivingArea", "LotSizeArea"])

ingest_test = Pipeline([ 
    ("ingestor", ingestor),
    ("fred_ingestor", fred_ingestor),
    ("cleaner", cleaner),
    ("flagger", flagger),
    ("market_maker", market_maker),
    ("cleanup", cleanup),
    ("null", null),
    ("dist_merger", dist_merger),
    ("iqr_flagger", iqr_flagger)
])


df_sold, df_listings = ingest_test.fit_transform(X=None, y=None)

In [3]:
display(df_listings.head())

,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,PropertyType,LivingArea,ListPrice,...,non_cali_coords_flag,price_ratio,price_per_sqft,days_on_market,yr_month,listing_to_contract_days,contract_to_close_days,index_right,DistrictNa,outlier_flag
446,1175000.0,1076078118,2024-06-04,1165000.0,32.854896,-117.233948,8258 CAMINITO MODENA,Residential,1500.0,1175000.0,...,False,0.991489,776.666667,27.0,2024-06,5.0,22.0,572.0,San Diego Unified,False
616,3300.0,1076012168,2024-06-02,3300.0,34.099080,-117.422367,17525 Arrow Boulevard,CommercialLease,NaN,3300.0,...,False,1.000000,NaN,2.0,2024-06,2.0,0.0,524.0,Fontana Unified,True
826,1200000.0,1075992389,2024-05-31,1200000.0,33.974734,-117.365312,2826 10th Street,ResidentialIncome,NaN,1200000.0,...,False,1.000000,NaN,0.0,2024-05,0.0,0.0,482.0,Riverside Unified,False
949,4000.0,1075988218,2024-06-05,4000.0,34.224808,-118.558653,8502 Belmar Avenue,ResidentialLease,1500.0,4000.0,...,False,1.000000,2.666667,5.0,2024-06,4.0,1.0,281.0,Los Angeles Unified,False
1271,1595000.0,1075984118,2024-05-31,1595000.0,33.688540,-116.385903,49220 Quercus Lane,Residential,2381.0,1595000.0,...,False,1.000000,669.886602,1.0,2024-05,1.0,0.0,472.0,Desert Sands Unified,False


In [4]:
display(df_listings.info())

<class 'pandas.core.frame.DataFrame'>
Index: 253027 entries, 446 to 893479
Data columns (total 63 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   OriginalListPrice            253027 non-null  float64       
 1   ListingKey                   253027 non-null  int64         
 2   CloseDate                    253027 non-null  datetime64[ns]
 3   ClosePrice                   253027 non-null  float64       
 4   Latitude                     253027 non-null  float64       
 5   Longitude                    253027 non-null  float64       
 6   UnparsedAddress              252650 non-null  object        
 7   PropertyType                 253027 non-null  object        
 8   LivingArea                   240572 non-null  float64       
 9   ListPrice                    253024 non-null  float64       
 10  DaysOnMarket                 253027 non-null  int64         
 11  ListOfficeName               

None

# something

In [5]:
# TODO : save a df with bad data removed and one with all the flags still
# Include a written comparison of dataset size and median values before and after filtering. OK 

In [6]:
def drop_flagged(df, flag_columns):
    """
    Drops rows from the DataFrame where any of the specified flag columns are True.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame.
    flag_columns (list): List of column names that contain boolean flags.
    
    Returns:
    pd.DataFrame: A new DataFrame with flagged rows removed.
    """
    # Create a boolean mask where any of the flag columns are True
    mask = df[flag_columns].any(axis=1)
    
    # Return a new DataFrame with flagged rows removed
    return df[~mask]

In [7]:
flag_columns = [col for col in df_listings.columns if 'flag' in col.lower()]

df_sold_cleared = drop_flagged(df_sold, flag_columns)
df_listings_cleared = drop_flagged(df_listings, flag_columns)


In [8]:
display(df_sold_cleared.info())
display(df_sold_cleared.shape)

<class 'pandas.core.frame.DataFrame'>
Index: 689609 entries, 0 to 681594
Data columns (total 74 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   Flooring                     402912 non-null  object        
 1   ViewYN                       623498 non-null  object        
 2   WaterfrontYN                 393 non-null     object        
 3   BasementYN                   10304 non-null   object        
 4   PoolPrivateYN                607149 non-null  object        
 5   OriginalListPrice            689609 non-null  float64       
 6   ListingKey                   689609 non-null  int64         
 7   CloseDate                    689609 non-null  datetime64[ns]
 8   ClosePrice                   689609 non-null  float64       
 9   Latitude                     689609 non-null  float64       
 10  Longitude                    689609 non-null  float64       
 11  UnparsedAddress              68

None

(689609, 74)

In [9]:
display(df_listings_cleared.info())
display(df_listings_cleared.shape)

<class 'pandas.core.frame.DataFrame'>
Index: 217210 entries, 446 to 893479
Data columns (total 63 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   OriginalListPrice            217210 non-null  float64       
 1   ListingKey                   217210 non-null  int64         
 2   CloseDate                    217210 non-null  datetime64[ns]
 3   ClosePrice                   217210 non-null  float64       
 4   Latitude                     217210 non-null  float64       
 5   Longitude                    217210 non-null  float64       
 6   UnparsedAddress              216881 non-null  object        
 7   PropertyType                 217210 non-null  object        
 8   LivingArea                   207911 non-null  float64       
 9   ListPrice                    217207 non-null  float64       
 10  DaysOnMarket                 217210 non-null  int64         
 11  ListOfficeName               

None

(217210, 63)

In [12]:
print("Original df_sold shape:", df_sold.shape)
print("Cleared df_sold shape:", df_sold_cleared.shape)
print("Original df_listings shape:", df_listings.shape)
print("Cleared df_listings shape:", df_listings_cleared.shape)

Original df_sold shape: (803696, 74)
Cleared df_sold shape: (689609, 74)
Original df_listings shape: (253027, 63)
Cleared df_listings shape: (217210, 63)


In [13]:
df_sold_cleared.to_csv("../data/processed/df_sold_cleared.csv", index=False)
df_listings_cleared.to_csv("../data/processed/df_listings_cleared.csv", index=False)
df_sold.to_csv("../data/processed/df_sold.csv", index=False)
df_listings.to_csv("../data/processed/df_listings.csv", index=False)